# 06 — Evaluate: Test set · Confusion Matrix · Learning Curves · Occlusion · Robustness · Speed · Business KPI

**หน้าที่ของ notebook นี้ (MASTER PROMPT ข้อ 7 + ข้อ 9 — ห้ามข้าม)**
1. ประเมินทั้ง Baseline และ Improved บน **test set** → per-class AP, P/R/F1, mAP50/50-95 → เลือก **โมเดลสุดท้าย** (กติกา: test mAP50-95 สูงสุด; ต่างกัน < 0.5% ให้ตัดสินด้วย recall ของ overripe)
2. **Confusion matrix** raw + normalized + วิเคราะห์เป็นข้อความอัตโนมัติ
3. **Learning curves** (box / cls / l1 loss train vs val, mAP50, mAP50-95, P, R) + วินิจฉัย overfit / underfit อัตโนมัติ — *YOLO26 ใช้ L1 loss แทน DFL (reg_max=1) จึงไม่มี dfl_loss ตามที่ spec เขียน*
4. **PR curve + F1-confidence curve** → conf threshold สำหรับ deploy
5. **Occlusion test** (Paper 1): full test vs occlusion vs clear subset, sweep IoU × conf, **NMS vs NMS-free** ของ YOLO26
6. **Robustness test** (Paper 2): low-light / over-exposure / motion blur / tilt / noise → ตาราง mAP drop
7. **Speed benchmark** (Paper 3): FPS, ms/frame บน GPU (FP32/FP16) และ CPU, model size, mAP–speed trade-off
8. **Business KPI จำลอง** (Paper 3): ผล/นาที, อัตราคัดแยกผิด, ประมาณการลด food loss — สมมติฐานทั้งหมดอยู่ใน `NB06["kpi"]` (ปรับได้)
9. `reports/06_*.{json,csv,md}` + `figures/06_*.png` (dpi 300) + `final_model` ลง `configs/project.yaml` ให้ nb07

> kernel: **ML Environment (ds69) + YOLO26** · ต้องรัน nb01–05 ก่อน (ใช้ `runs/detect/*/weights/best.pt`)


In [ ]:
# === CELL 1 === โหลด config + best.pt ทั้ง 2 + helper สำหรับ val/predict ด้วย API ของ ultralytics 8.4 (nms=None / nms=False)
# ทำอะไร : อ่าน path จาก project.yaml (nb04: data yaml, nb05: experiments), โหลดโมเดล, รวมค่าตั้งของการประเมินทั้งหมดไว้ใน NB06
# ทำไม   : ทุกตัวเลขในเล่มรายงานต้องมาจากที่นี่ที่เดียว และ NMS-free ของ YOLO26 ต้องเรียกด้วย nms=False (ไม่ใช่ end2end= ตาม spec — nb01 ตรวจแล้ว)
import os, sys, json, time, shutil, random, importlib, math
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import yaml
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
import ultralytics

PROJECT_ROOT = Path.cwd().resolve()
CFG_PATH = PROJECT_ROOT / "configs" / "project.yaml"
assert CFG_PATH.exists(), f"ไม่พบ {CFG_PATH} → รัน 01_setup_env.ipynb ก่อน"
with open(CFG_PATH, encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)
DIRS = {k: PROJECT_ROOT / v for k, v in CONFIG["dirs"].items()}
SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
CLASSES = {int(k): v for k, v in CONFIG["classes"].items()}
NAMES = list(CLASSES.values())
NC = len(CLASSES)
assert "experiments" in CONFIG and {"baseline", "improved"} <= set(CONFIG["experiments"]), "ไม่พบ experiments ใน project.yaml → รัน 05_train_yolo26s.ipynb ให้จบก่อน"
assert "data_yaml" in CONFIG, "ไม่พบ data_yaml → รัน nb04"
sys.path.insert(0, str(DIRS["src"]))
import viz
importlib.reload(viz)
viz.apply_style()
CLASS_RGB = viz.class_colors(CONFIG)

QUICK_TEST = os.environ.get("NB06_QUICK_TEST", "0") == "1"    # โหมดทดสอบโค้ด (ภาพเล็ก/ตัวอย่างน้อย) — ไม่ใช่ผลจริง
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"[OK] device = {DEVICE!r} ({torch.cuda.get_device_name(0) if DEVICE != 'cpu' else 'CPU'}) | ultralytics {ultralytics.__version__} | torch {torch.__version__}")

# --- ค่าตั้งการประเมินทั้งหมด ---
NB06 = {
    "imgsz": CONFIG["model"]["imgsz"], "batch": 16,
    "iou_default": 0.7,                 # NMS IoU default ของ ultralytics (ใช้กับ nms=None)
    "occl_iou_sweep": [0.5, 0.6, 0.7, 0.8],          # sweep NMS IoU บน occlusion subset (Paper 1)
    "occl_conf_sweep": [0.10, 0.25, 0.40, 0.55],     # sweep confidence บน occlusion subset
    "count_conf": 0.25,                 # conf สำหรับนับจำนวน box (NMS vs NMS-free)
    "dup_iou": 0.7,                     # คู่ prediction คลาสเดียวกันที่ IoU > 0.7 ถือว่าเป็น "box ซ้ำ"
    "robust": {                         # การรบกวนจำลอง (Paper 2) — ทำบนภาพ test ทุกใบ
        "low_light":     {"alpha": 0.40, "beta": 0},     # ลดความสว่างเหลือ 40%
        "over_exposure": {"alpha": 1.50, "beta": 60},    # สว่างจ้า + เลื่อนขึ้น 60
        "motion_blur":   {"ksize": 15},                  # เบลอเชิงเส้นแนวนอน 15 px (สายพานเคลื่อน)
        "tilt":          {"degrees": 20},                # หมุน 20° (หมุน bbox ตามและครอบด้วยกล่องแกนตั้ง)
        "noise":         {"sigma": 25},                  # Gaussian noise σ=25 (เซ็นเซอร์กล้อง/แสงน้อย)
    },
    "speed": {"n_images": 100, "warmup": 10, "cpu_images": 30},
    "kpi": {   # *** สมมติฐานสำหรับ Business KPI (ตกลง 2026-09-20 ว่าใช้ค่าสมมติ) — ปรับได้ที่นี่ที่เดียว ***
        "line_speed_bpm": 120,          # กล้วยที่ผ่านจุดตรวจ ต่อนาที
        "bananas_per_frame": 1,         # จำนวนผลต่อเฟรมที่ระบบต้องตัดสิน (ประมาณการแบบระมัดระวัง)
        "cost_per_banana_thb": 1.5,     # ต้นทุนวัตถุดิบต่อผล (บาท)
        "human_error_rate": 0.05,       # อัตราคัดแยกผิดของคน (5%) — สมมติ
        "shift_hours": 8, "days_per_month": 26,
        "loss_fraction_of_misgraded": 0.5,   # สัดส่วนของกล้วยที่คัดผิดแล้วกลายเป็น food loss (สมมติ)
    },
}
if QUICK_TEST:
    NB06.update(imgsz=320)
    NB06["occl_iou_sweep"], NB06["occl_conf_sweep"] = [0.5, 0.7], [0.25, 0.5]
    NB06["speed"] = {"n_images": 8, "warmup": 2, "cpu_images": 8}
    NB06["robust"] = {k: NB06["robust"][k] for k in ("low_light", "tilt")}
EVAL_DIR = DIRS["runs"] / ("eval_quicktest" if QUICK_TEST else "eval")
DATA_TEST = str(PROJECT_ROOT / CONFIG["data_yaml"]["baseline"])
DATA_OCC = str(PROJECT_ROOT / CONFIG["data_yaml"]["test_occlusion"])
DATA_CLEAR = str(PROJECT_ROOT / CONFIG["data_yaml"]["test_clear"])
DS = DIRS["datasets"]
EXPS = {name: PROJECT_ROOT / rec["best"] for name, rec in CONFIG["experiments"].items()}
for name, p in EXPS.items():
    assert p.exists(), f"ไม่พบ {p} → รัน nb05 ให้จบ"
MODELS = {name: YOLO(str(p)) for name, p in EXPS.items()}
RUN = {"notebook": "06_evaluate", "started_at": datetime.now().isoformat(timespec="seconds"), "quick_test": QUICK_TEST, "settings": NB06}

def val(name, data, tag, split="val", nms=None, conf=None, iou=None, plots=False):
    # ประเมินด้วย ultralytics: nms=None = one-to-many head + NMS ภายนอก (default) | nms=False = NMS-free one-to-one head (YOLO26)
    kw = dict(data=data, split=split, imgsz=NB06["imgsz"], batch=NB06["batch"], device=DEVICE, nms=nms, plots=plots, verbose=False,
              project=str(EVAL_DIR), name=f"{name}_{tag}", exist_ok=True)
    if conf is not None:
        kw["conf"] = conf
    if iou is not None:
        kw["iou"] = iou
    return MODELS[name].val(**kw)

def summarize(m):
    # ดึงตัวเลขจาก DetMetrics เป็น dict (ค่า per-class อยู่ตามลำดับ ap_class_index)
    b = m.box
    idx = list(b.ap_class_index)
    per = {NAMES[c]: {"AP50": float(b.ap50[i]), "AP50-95": float(b.ap[i]), "P": float(b.p[i]), "R": float(b.r[i]), "F1": float(b.f1[i])}
           for i, c in enumerate(idx)}
    return {"mAP50": float(b.map50), "mAP50-95": float(b.map), "mAP75": float(b.map75), "P": float(b.mp), "R": float(b.mr),
            "F1": float(np.mean(b.f1)) if len(b.f1) else 0.0, "per_class": per,
            "speed_ms": {k: round(float(v), 2) for k, v in m.speed.items()}}

print(f"[OK] models: " + " | ".join(f"{k} ← {v.relative_to(PROJECT_ROOT)}" for k, v in EXPS.items()))
print(f"[OK] test yaml: {Path(DATA_TEST).name} | occlusion: {Path(DATA_OCC).name} | clear: {Path(DATA_CLEAR).name} | imgsz {NB06['imgsz']} | eval → {EVAL_DIR.relative_to(PROJECT_ROOT)}")


In [ ]:
# === CELL 2 === ประเมินบน TEST set ทั้ง 2 โมเดล → per-class AP table + ตารางเทียบ Baseline vs Improved + เลือกโมเดลสุดท้าย
# ทำอะไร : model.val(split="test") (plots=True เพื่อให้ ultralytics คำนวณ confusion matrix ด้วย) → ตาราง + กราฟ AP ต่อคลาส
# ทำไม   : valid ถูกใช้เลือก best epoch ไปแล้ว → ตัวเลขที่รายงานได้ต้องมาจาก test ที่โมเดลไม่เคยเห็น (spec ข้อ 9)
TEST = {}
for name in MODELS:
    m = val(name, DATA_TEST, "test", split="test", plots=True)
    TEST[name] = {"metrics": m, "summary": summarize(m)}
    s = TEST[name]["summary"]
    print(f"[TEST] {name:<9} mAP50 {s['mAP50']:.4f} | mAP50-95 {s['mAP50-95']:.4f} | mAP75 {s['mAP75']:.4f} | P {s['P']:.4f} | R {s['R']:.4f} | F1 {s['F1']:.4f}")

rows = []
for name, t in TEST.items():
    for cls, v in t["summary"]["per_class"].items():
        rows.append({"model": name, "class": cls, **{k: round(x, 4) for k, x in v.items()}})
per_class = pd.DataFrame(rows).set_index(["model", "class"])
print("\n— per-class (test) —")
print(per_class.unstack(0).round(4).to_string())
comp = pd.DataFrame({name: {k: v for k, v in t["summary"].items() if k in ("mAP50", "mAP50-95", "mAP75", "P", "R", "F1")} for name, t in TEST.items()}).T
comp["overripe_recall"] = [TEST[n]["summary"]["per_class"].get("overripe", {}).get("R", np.nan) for n in comp.index]
print("\n— Baseline vs Improved (test) —")
print(comp.round(4).to_string())

# --- เลือกโมเดลสุดท้าย: test mAP50-95 สูงสุด; ต่างกัน < 0.005 → ใช้ recall ของ overripe (คลาส QC สำคัญ) ตัดสิน ---
names = list(TEST)
d = comp.loc["improved", "mAP50-95"] - comp.loc["baseline", "mAP50-95"]
if abs(d) >= 0.005:
    FINAL = "improved" if d > 0 else "baseline"
    reason = f"test mAP50-95 ต่างกัน {d:+.4f} (≥ 0.005)"
else:
    top = comp[comp["overripe_recall"] == comp["overripe_recall"].max()]      # โมเดลที่ overripe recall สูงสุด (อาจเสมอกัน)
    if len(top) == 1:
        FINAL = top.index[0]
        reason = f"test mAP50-95 ต่างกันเพียง {d:+.4f} (< 0.005) → ตัดสินด้วย overripe recall ({comp['overripe_recall'].round(4).to_dict()})"
    else:                                                                       # เสมอสองชั้น → ใช้ F1 (สมดุล P/R ทุกคลาส) เป็นเกณฑ์ที่ 3
        FINAL = top["F1"].idxmax()
        reason = (f"test mAP50-95 ต่างกันเพียง {d:+.4f} (< 0.005) และ overripe recall เสมอกัน "
                  f"({comp['overripe_recall'].round(4).to_dict()}) → ตัดสินด้วย F1 ({comp['F1'].round(4).to_dict()})")
OTHER = [n for n in names if n != FINAL][0]
print(f"\n[FINAL MODEL] {FINAL}  — เหตุผล: {reason}")
per_class.to_csv(DIRS["reports"] / "06_per_class_ap_test.csv", encoding="utf-8-sig")
RUN["test"] = {name: t["summary"] for name, t in TEST.items()}
RUN["final_model"] = {"name": FINAL, "reason": reason}

# --- กราฟ AP ต่อคลาส (AP50 และ AP50-95) ทั้ง 2 โมเดล ---
colors = {"baseline": "#9a9993", "improved": viz.SOURCE_COLORS["mendeley"]}
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(NC)
for ax, key in zip(axes, ("AP50", "AP50-95")):
    for i, name in enumerate(names):
        vals = [TEST[name]["summary"]["per_class"].get(c, {}).get(key, 0) for c in NAMES]
        bars = ax.bar(x + (i - 0.5) * 0.36, vals, width=0.34, color=colors[name], label=name)
        for b_, v in zip(bars, vals):
            ax.text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, color=viz.TEXT["secondary"])
    ax.set_xticks(x, NAMES)
    ax.set_ylim(0, 1.12)
    ax.set_title(f"{key} per class (test)", loc="left", fontsize=11)
handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle(f"Test set — {len(TEST[FINAL]['metrics'].box.ap_class_index)} classes · final model = {FINAL}", x=0.01, ha="left", fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.94))
fig.legend(handles, labels, loc="upper right", ncol=2, bbox_to_anchor=(0.99, 0.995))
out = viz.save_fig(fig, DIRS["figures"], "06_per_class_ap_test.png")
plt.show()
plt.close(fig)
print(f"[OK] {out.relative_to(PROJECT_ROOT)} | reports/06_per_class_ap_test.csv")


In [ ]:
# === CELL 3 === Confusion matrix (raw + normalized) ของทั้ง 2 โมเดล + วิเคราะห์เป็นข้อความ
# ทำอะไร : ใช้ confusion_matrix ที่ ultralytics คำนวณตอน val (conf 0.25, IoU 0.45; แถว = predicted, คอลัมน์ = true, ช่องสุดท้าย = background)
#          วาด heatmap แบบอ่านง่าย (raw count + normalized ตามคลาสจริง) และสรุปคู่คลาสที่สับสนมากสุดเป็นข้อความ
# ทำไม   : spec ข้อ 9 ต้องการทั้ง 2 แบบ + คำอธิบายว่าคลาสไหนสับสนกันเพราะอะไร (คาด semi_ripe ↔ ripe: สีต่อเนื่องกัน ไม่มีเส้นแบ่งชัด)
LABELS = NAMES + ["background"]

def cm_figure(name, path):
    M = TEST[name]["metrics"].confusion_matrix.matrix.astype(float)      # (nc+1, nc+1): [pred, true]
    assert M.shape == (NC + 1, NC + 1), f"confusion matrix shape {M.shape} ผิดคาด"
    col = M.sum(axis=0, keepdims=True)
    N = np.divide(M, col, out=np.zeros_like(M), where=col > 0)            # normalize ต่อคลาสจริง (คอลัมน์รวม = 1)
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.4))
    for ax, mat, title, fmt in ((axes[0], M, "raw counts", "{:.0f}"), (axes[1], N, "normalized by true class (column = 1.0)", "{:.2f}")):
        im = ax.imshow(mat, cmap="Blues", vmin=0, vmax=(mat.max() if title.startswith("raw") else 1.0))
        ax.set_xticks(range(NC + 1), LABELS, rotation=30, ha="right")
        ax.set_yticks(range(NC + 1), LABELS)
        ax.set_xlabel("True")
        ax.set_ylabel("Predicted")
        ax.set_title(f"{name} — {title}", loc="left", fontsize=11)
        ax.grid(False)
        thr = (mat.max() if title.startswith("raw") else 1.0) * 0.55
        for i in range(NC + 1):
            for j in range(NC + 1):
                if mat[i, j] > 0 or i == j:
                    ax.text(j, i, fmt.format(mat[i, j]), ha="center", va="center", fontsize=9, color="white" if mat[i, j] > thr else viz.TEXT["primary"])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03)
    fig.suptitle(f"Confusion matrix — {name} (test, conf 0.25, IoU 0.45)", x=0.01, ha="left", fontsize=12)
    fig.tight_layout()
    out = viz.save_fig(fig, DIRS["figures"], path)
    plt.show()
    plt.close(fig)
    return M, N, out

def cm_analysis(name, M, N):
    # สร้างข้อความสรุปอัตโนมัติ: ต่อคลาสจริง → ถูกกี่ %, สับสนไปคลาสไหนมากสุด, พลาดตรวจ (background) กี่ %
    lines = []
    total_gt = M[:, :NC].sum()
    correct = np.trace(M[:NC, :NC])
    lines.append(f"[{name}] ตรวจจับได้และจัดคลาสถูก {correct:.0f}/{total_gt:.0f} กล่องจริง ({correct / max(total_gt, 1) * 100:.1f}%) | "
                 f"พลาดตรวจ (missed) {M[NC, :NC].sum():.0f} | ตรวจเจอของที่ไม่มี (false alarm) {M[:NC, NC].sum():.0f}")
    pairs = []
    for j, true in enumerate(NAMES):
        colsum = M[:, j].sum()
        if colsum == 0:
            continue
        off = [(N[i, j], LABELS[i]) for i in range(NC + 1) if i != j]
        worst, worst_name = max(off)
        lines.append(f"  true {true:<10}: ถูก {N[j, j] * 100:5.1f}% | สับสนมากสุด → {worst_name} {worst * 100:.1f}% | missed {N[NC, j] * 100:.1f}%")
        for i in range(NC):
            if i != j and M[i, j] > 0:
                pairs.append((N[i, j], true, NAMES[i], int(M[i, j])))
    pairs.sort(reverse=True)
    if pairs:
        lines.append("  คู่คลาสที่สับสนมากสุด (true → predicted): " + "; ".join(f"{t}→{p} {r * 100:.1f}% ({n})" for r, t, p, n in pairs[:3]))
        top = pairs[0]
        adjacent = abs(NAMES.index(top[1]) - NAMES.index(top[2])) == 1
        lines.append("  ตีความ: " + ("คู่ที่สับสนคือระดับความสุกที่ติดกัน — สีเปลี่ยนต่อเนื่อง (เขียว→เหลืองอมเขียว→เหลือง→น้ำตาล) จึงไม่มีเส้นแบ่งชัด "
                                   "และ label ของมนุษย์เองก็ไม่คงเส้นคงวาบริเวณรอยต่อ" if adjacent else
                                   "คู่ที่สับสนไม่ใช่ระดับติดกัน — ควรตรวจ label/แสงของภาพกลุ่มนี้เพิ่ม"))
    return lines

CM = {}
for name in MODELS:
    M, N, out = cm_figure(name, f"06_confusion_matrix_{name}.png")
    pd.DataFrame(M.astype(int), index=[f"pred_{l}" for l in LABELS], columns=[f"true_{l}" for l in LABELS]).to_csv(DIRS["reports"] / f"06_confusion_{name}.csv", encoding="utf-8-sig")
    CM[name] = {"raw": M, "norm": N, "analysis": cm_analysis(name, M, N)}
    print("\n".join(CM[name]["analysis"]))
    print(f"[OK] {out.relative_to(PROJECT_ROOT)} | reports/06_confusion_{name}.csv\n")
RUN["confusion"] = {name: {"raw": CM[name]["raw"].astype(int).tolist(), "analysis": CM[name]["analysis"]} for name in CM}


In [ ]:
# === CELL 4 === Learning curves จาก results.csv ของทั้ง 2 run + วินิจฉัย overfit / underfit อัตโนมัติ
# ทำอะไร : plot box/cls/l1 loss (train vs val), mAP50, mAP50-95, precision, recall ตาม epoch → คำนวณ gap train-val, slope ช่วงท้าย, ตำแหน่ง best epoch
# ทำไม   : spec ข้อ 9 — และ YOLO26 ไม่มี dfl_loss (reg_max=1 → ใช้ L1 regression loss แทน) จึง plot l1_loss พร้อมระบุในกราฟ
curves = {}
for name in MODELS:
    p = DIRS["reports"] / f"05_results_{name}.csv"
    if not p.exists():
        p = PROJECT_ROOT / CONFIG["experiments"][name]["run_dir"] / "results.csv"
    c = pd.read_csv(p)
    c.columns = [k.strip() for k in c.columns]
    curves[name] = c
loss_keys = [k for k in ("box_loss", "cls_loss", "l1_loss", "dfl_loss") if f"train/{k}" in next(iter(curves.values())).columns]
metric_keys = [("metrics/mAP50(B)", "mAP50"), ("metrics/mAP50-95(B)", "mAP50-95"), ("metrics/precision(B)", "precision"), ("metrics/recall(B)", "recall")]

fig, axes = plt.subplots(2, 4, figsize=(16, 7.2))
axes = axes.reshape(-1)
for i, k in enumerate(loss_keys):
    ax = axes[i]
    for name, c in curves.items():
        ax.plot(c["epoch"], c[f"train/{k}"], color=colors[name], lw=2, label=f"{name} train")
        ax.plot(c["epoch"], c[f"val/{k}"], color=colors[name], lw=1.4, ls="--", label=f"{name} val")
    ax.set_title(f"{k} (solid train / dashed val)" + (" — YOLO26 uses L1 instead of DFL" if k == "l1_loss" else ""), loc="left", fontsize=9.5)
    ax.set_xlabel("epoch")
for j, (col, label) in enumerate(metric_keys):
    ax = axes[len(loss_keys) + j]
    for name, c in curves.items():
        ax.plot(c["epoch"], c[col], color=colors[name], lw=2, label=name)
        bi = int(c["metrics/mAP50-95(B)"].idxmax())
        ax.scatter([c.loc[bi, "epoch"]], [c.loc[bi, col]], color=colors[name], s=30, zorder=5)
    ax.set_title(f"val {label} (dot = best epoch)", loc="left", fontsize=9.5)
    ax.set_xlabel("epoch")
for ax in axes[len(loss_keys) + len(metric_keys):]:
    ax.axis("off")
axes[0].legend(fontsize=7.5)
axes[len(loss_keys)].legend(fontsize=7.5)
fig.suptitle(f"Learning curves — yolo26s, {CONFIG['model'].get('epochs_actual', '?')} epochs, MuSGD, seed {SEED}", x=0.01, ha="left", fontsize=12)
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "06_learning_curves.png")
plt.show()
plt.close(fig)

def diagnose(name, c):
    # กติกา: (1) val loss ช่วง 10 epoch ท้ายมีแนวโน้มขึ้นขณะ train loss ลง + best epoch อยู่ก่อน 70% → overfit
    #        (2) best epoch อยู่ใน 3 epoch สุดท้าย และ val mAP50-95 ยังไต่ขึ้น (> 0.0005/epoch) → under-trained (ยังไม่อิ่มตัว)
    #        (3) นอกนั้น → converged
    n = len(c)
    tail = c.tail(min(10, n))
    ep = tail["epoch"].to_numpy(dtype=float)
    slope = lambda s: float(np.polyfit(ep, s.to_numpy(dtype=float), 1)[0]) if len(ep) > 1 else 0.0
    map_slope = slope(tail["metrics/mAP50-95(B)"])
    val_up = sum(slope(tail[f"val/{k}"]) > 0 for k in loss_keys)
    train_down = sum(slope(tail[f"train/{k}"]) < 0 for k in loss_keys)
    best_i = int(c["metrics/mAP50-95(B)"].idxmax())
    best_ep, best_map = int(c.loc[best_i, "epoch"]), float(c.loc[best_i, "metrics/mAP50-95(B)"])
    gaps = {k: float(c[f"val/{k}"].iloc[-1] - c[f"train/{k}"].iloc[-1]) for k in loss_keys}
    if val_up >= 2 and train_down >= 2 and best_ep < 0.7 * n:
        verdict = "OVERFIT — val loss กลับขึ้นขณะ train loss ยังลง และ best epoch อยู่ค่อนต้น"
    elif best_ep >= n - 3 and map_slope > 0.0005:
        verdict = f"UNDER-TRAINED — best epoch = {best_ep}/{n} และ val mAP50-95 ยังไต่ขึ้น {map_slope * 10:+.4f}/10 epoch → epoch เพิ่มน่าจะได้ผลดีขึ้น (spec เดิม 150)"
    else:
        verdict = "CONVERGED — val mAP50-95 ทรงตัว, val loss ไม่กลับขึ้น"
    txt = (f"[{name}] best epoch {best_ep}/{n} (val mAP50-95 {best_map:.4f}) | slope mAP50-95 10 epoch ท้าย {map_slope * 10:+.4f} | "
           f"gap val−train ล่าสุด: " + ", ".join(f"{k} {g:+.3f}" for k, g in gaps.items()) + f" | val loss ขึ้น {val_up}/{len(loss_keys)} ตัว → {verdict}")
    return txt, {"best_epoch": best_ep, "epochs": n, "best_val_map50_95": best_map, "map_slope_per_10ep": round(map_slope * 10, 5), "loss_gap": gaps, "verdict": verdict}

RUN["learning_curves"] = {}
for name, c in curves.items():
    txt, rec = diagnose(name, c)
    RUN["learning_curves"][name] = rec
    print(txt)
print(f"[OK] {out.relative_to(PROJECT_ROOT)} (loss ที่ plot: {loss_keys})")


In [ ]:
# === CELL 5 === PR curve + F1-confidence curve (ต่อคลาส + เฉลี่ย) ของโมเดลสุดท้าย → conf threshold สำหรับ deploy
# ทำอะไร : อ่านเส้นโค้งจาก DetMetrics.box.curves_results ([x, y(nc,1000), xlabel, ylabel]) ที่ ultralytics คำนวณตอน val test
#          → เลือก conf ที่ F1 เฉลี่ยสูงสุดเป็นค่า deploy (nb07)
# ทำไม   : spec ข้อ 9 ต้องการทั้ง 2 กราฟ และ conf ที่ใช้จริงต้องมีที่มา ไม่ใช่ 0.25 ลอย ๆ
def get_curve(m, xlabel, ylabel):
    for x, y, xl, yl in m.box.curves_results:
        if xl == xlabel and yl == ylabel:
            return np.asarray(x), np.asarray(y)
    raise KeyError((xlabel, ylabel))

DEPLOY = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
for name in [FINAL, OTHER]:
    m = TEST[name]["metrics"]
    idx = list(m.box.ap_class_index)
    rx, ry = get_curve(m, "Recall", "Precision")      # PR curve
    fx, fy = get_curve(m, "Confidence", "F1")         # F1 vs confidence
    f1_mean = fy.mean(axis=0)
    bi = int(f1_mean.argmax())
    DEPLOY[name] = {"conf": round(float(fx[bi]), 3), "f1": round(float(f1_mean[bi]), 4)}
    lw, alpha = (2.2, 1.0) if name == FINAL else (1.2, 0.45)
    for i, c in enumerate(idx):
        axes[0].plot(rx, ry[i], color=CLASS_RGB[c], lw=lw, alpha=alpha, label=f"{NAMES[c]} AP50 {m.box.ap50[i]:.3f}" if name == FINAL else None)
        axes[1].plot(fx, fy[i], color=CLASS_RGB[c], lw=lw, alpha=alpha, label=NAMES[c] if name == FINAL else None)
    axes[0].plot(rx, ry.mean(axis=0), color=viz.TEXT["primary"] if name == FINAL else "#9a9993", lw=lw + 0.5, ls="-" if name == FINAL else ":",
                 label=f"{name} mean (mAP50 {m.box.map50:.3f})")
    axes[1].plot(fx, f1_mean, color=viz.TEXT["primary"] if name == FINAL else "#9a9993", lw=lw + 0.5, ls="-" if name == FINAL else ":",
                 label=f"{name} mean F1 (best {f1_mean[bi]:.3f} @ conf {fx[bi]:.2f})")
    if name == FINAL:
        axes[1].axvline(fx[bi], color=viz.TEXT["secondary"], ls="--", lw=1)
axes[0].set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02))
axes[0].set_title(f"Precision–Recall (test) — final = {FINAL}, faded = {OTHER}", loc="left", fontsize=10.5)
axes[1].set(xlabel="Confidence threshold", ylabel="F1", xlim=(0, 1), ylim=(0, 1.02))
axes[1].set_title("F1 vs confidence (test) — dashed = deploy threshold", loc="left", fontsize=10.5)
axes[0].legend(fontsize=7.5, loc="lower left")
axes[1].legend(fontsize=7.5, loc="lower center")
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "06_pr_f1_curves.png")
plt.show()
plt.close(fig)
print(f"[OK] {out.relative_to(PROJECT_ROOT)}")
print(f"[DEPLOY] conf threshold ของ {FINAL} = {DEPLOY[FINAL]['conf']} (F1 เฉลี่ยสูงสุด {DEPLOY[FINAL]['f1']:.4f}) | {OTHER}: conf {DEPLOY[OTHER]['conf']} F1 {DEPLOY[OTHER]['f1']:.4f}")
RUN["deploy_threshold"] = DEPLOY


In [ ]:
# === CELL 6 === Occlusion test (Paper 1): full vs occlusion vs clear subset + sweep IoU/conf + NMS vs NMS-free ของ YOLO26
# ทำอะไร : val ทั้ง 2 โมเดลบน 3 ชุด → ตาราง; sweep NMS-IoU และ conf บน occlusion subset (โมเดลสุดท้าย);
#          เทียบ nms=None (one-to-many + NMS) กับ nms=False (one-to-one, NMS-free): mAP, เวลา postprocess, จำนวน box/ภาพ และ box ซ้ำ
# ทำไม   : spec ข้อ 7 Paper 1 — กล้วยซ้อนกันคือจุดที่ NMS ตัด box ถูกทิ้ง; YOLO26 อ้างว่า NMS-free แก้ได้ → ต้องพิสูจน์ด้วยตัวเลข
SUBSETS = {"full_test": (DATA_TEST, "test"), "occlusion": (DATA_OCC, "val"), "clear": (DATA_CLEAR, "val")}
occ_rows = []
for name in MODELS:
    for sub, (data, split) in SUBSETS.items():
        s = TEST[name]["summary"] if sub == "full_test" else summarize(val(name, data, sub, split=split))
        occ_rows.append({"model": name, "subset": sub, "mAP50": s["mAP50"], "mAP50-95": s["mAP50-95"], "P": s["P"], "R": s["R"]})
occ_tab = pd.DataFrame(occ_rows).set_index(["model", "subset"])
print("— full test vs occlusion vs clear (nms=None) —")
print(occ_tab.round(4).unstack(1).to_string())
for name in MODELS:
    d = occ_tab.loc[(name, "occlusion"), "mAP50"] - occ_tab.loc[(name, "clear"), "mAP50"]
    print(f"  {name}: mAP50 occlusion − clear = {d:+.4f}")

# --- sweep บน occlusion subset (โมเดลสุดท้าย) ---
iou_rows = [{"iou": i, **{k: v for k, v in summarize(val(FINAL, DATA_OCC, f"occ_iou{i}", iou=i)).items() if k in ("mAP50", "mAP50-95", "P", "R")}} for i in NB06["occl_iou_sweep"]]
conf_rows = [{"conf": c, **{k: v for k, v in summarize(val(FINAL, DATA_OCC, f"occ_conf{c}", conf=c)).items() if k in ("mAP50", "P", "R", "F1")}} for c in NB06["occl_conf_sweep"]]
iou_tab, conf_tab = pd.DataFrame(iou_rows).set_index("iou"), pd.DataFrame(conf_rows).set_index("conf")
print(f"\n— {FINAL}: sweep NMS IoU (occlusion subset, nms=None) —")
print(iou_tab.round(4).to_string())
print(f"\n— {FINAL}: sweep confidence (occlusion subset) —")
print(conf_tab.round(4).to_string())

# --- NMS vs NMS-free ---
nms_rows = []
for name in MODELS:
    for sub, (data, split) in (("full_test", SUBSETS["full_test"]), ("occlusion", SUBSETS["occlusion"])):
        for mode, nms_flag in (("NMS (nms=None)", None), ("NMS-free (nms=False)", False)):
            s = summarize(val(name, data, f"{sub}_{'nmsfree' if nms_flag is False else 'nms'}", split=split, nms=nms_flag))
            nms_rows.append({"model": name, "subset": sub, "mode": mode, "mAP50": s["mAP50"], "mAP50-95": s["mAP50-95"],
                             "postprocess_ms": s["speed_ms"]["postprocess"], "inference_ms": s["speed_ms"]["inference"]})
nms_tab = pd.DataFrame(nms_rows).set_index(["model", "subset", "mode"])
print("\n— NMS vs NMS-free (val) —")
print(nms_tab.round(4).to_string())

# จำนวน box ต่อภาพ + box ซ้ำ บน occlusion subset (conf 0.25) — วัดว่า NMS-free ทิ้ง/ซ้ำมากน้อยแค่ไหนเทียบ GT
def iou_xyxy(a, b):
    iw, ih = max(0.0, min(a[2], b[2]) - max(a[0], b[0])), max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0.0
occ_files = [l.strip() for l in (DS / "test_occlusion.txt").read_text(encoding="utf-8").splitlines() if l.strip()]
gt_boxes = sum(sum(1 for ln in (Path(f).parent.parent / "labels" / (Path(f).stem + ".txt")).read_text().splitlines() if ln.strip()) for f in occ_files)
count_rows = []
for mode, nms_flag in (("NMS", None), ("NMS-free", False)):
    n_pred, n_dup = 0, 0
    for r in MODELS[FINAL].predict(occ_files, imgsz=NB06["imgsz"], device=DEVICE, conf=NB06["count_conf"], nms=nms_flag, verbose=False, stream=True):
        xyxy, cls = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy()
        n_pred += len(xyxy)
        n_dup += sum(1 for i in range(len(xyxy)) for j in range(i + 1, len(xyxy)) if cls[i] == cls[j] and iou_xyxy(xyxy[i], xyxy[j]) > NB06["dup_iou"])
    count_rows.append({"mode": mode, "pred_boxes": n_pred, "gt_boxes": gt_boxes, "pred/gt": round(n_pred / max(gt_boxes, 1), 3), "duplicate_pairs(IoU>0.7)": n_dup})
count_tab = pd.DataFrame(count_rows).set_index("mode")
print(f"\n— {FINAL} บน occlusion subset ({len(occ_files)} ภาพ, conf {NB06['count_conf']}): จำนวน box —")
print(count_tab.to_string())

# --- กราฟ: (1) mAP50 ต่อ subset ทั้ง 2 โมเดล (2) sweep IoU (3) sweep conf (4) NMS vs NMS-free บน occlusion ---
fig, axes = plt.subplots(1, 4, figsize=(17, 3.9))
subs = list(SUBSETS)
for i, name in enumerate(MODELS):
    vals = [occ_tab.loc[(name, s_), "mAP50"] for s_ in subs]
    bars = axes[0].bar(np.arange(3) + (i - 0.5) * 0.36, vals, width=0.34, color=colors[name], label=name)
    for b_, v in zip(bars, vals):
        axes[0].text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, color=viz.TEXT["secondary"])
axes[0].set_xticks(range(3), subs)
axes[0].set_ylim(0, 1.12)
axes[0].set_title("mAP50 by test subset", loc="left", fontsize=10)
axes[0].legend(fontsize=8)
axes[1].plot(iou_tab.index, iou_tab["mAP50"], marker="o", color=viz.SOURCE_COLORS["mendeley"], lw=2, label="mAP50")
axes[1].plot(iou_tab.index, iou_tab["mAP50-95"], marker="s", color=viz.SOURCE_COLORS["roboflow"], lw=2, label="mAP50-95")
axes[1].set(xlabel="NMS IoU threshold")
axes[1].set_title(f"{FINAL} — occlusion subset vs NMS IoU", loc="left", fontsize=10)
axes[1].legend(fontsize=8)
for k, col in (("P", viz.SOURCE_COLORS["mendeley"]), ("R", viz.SOURCE_COLORS["roboflow"]), ("F1", viz.TEXT["primary"])):
    axes[2].plot(conf_tab.index, conf_tab[k], marker="o", color=col, lw=2, label=k)
axes[2].set(xlabel="confidence threshold")
axes[2].set_title(f"{FINAL} — occlusion subset vs conf", loc="left", fontsize=10)
axes[2].legend(fontsize=8)
modes = ["NMS (nms=None)", "NMS-free (nms=False)"]
for i, name in enumerate(MODELS):
    vals = [nms_tab.loc[(name, "occlusion", m_), "mAP50"] for m_ in modes]
    bars = axes[3].bar(np.arange(2) + (i - 0.5) * 0.36, vals, width=0.34, color=colors[name], label=name)
    for b_, v in zip(bars, vals):
        axes[3].text(b_.get_x() + b_.get_width() / 2, b_.get_height(), f"{v:.3f}", ha="center", va="bottom", fontsize=7.5, color=viz.TEXT["secondary"])
axes[3].set_xticks(range(2), ["NMS", "NMS-free"])
axes[3].set_ylim(0, 1.12)
axes[3].set_title("occlusion subset — NMS vs NMS-free (mAP50)", loc="left", fontsize=10)
axes[3].legend(fontsize=8)
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "06_occlusion_nms.png")
plt.show()
plt.close(fig)
occ_tab.to_csv(DIRS["reports"] / "06_occlusion_subsets.csv", encoding="utf-8-sig")
nms_tab.to_csv(DIRS["reports"] / "06_nms_vs_nmsfree.csv", encoding="utf-8-sig")
RUN["occlusion"] = {"subsets": occ_tab.reset_index().to_dict("records"), "iou_sweep": iou_tab.reset_index().to_dict("records"),
                    "conf_sweep": conf_tab.reset_index().to_dict("records"), "nms_vs_nmsfree": nms_tab.reset_index().to_dict("records"),
                    "box_counts": count_tab.reset_index().to_dict("records")}
print(f"[OK] {out.relative_to(PROJECT_ROOT)} | reports/06_occlusion_subsets.csv, 06_nms_vs_nmsfree.csv")


In [ ]:
# === CELL 7 === Robustness test (Paper 2): สร้าง test set จำลอง 5 แบบ → ตาราง mAP drop เทียบ clean ทั้ง 2 โมเดล
# ทำอะไร : แปลงภาพ test ทุกใบ (cv2) → datasets/banana/robustness/<variant>/{images,labels} + yaml → val → mAP50/mAP50-95 + drop
#          tilt: หมุนภาพรอบจุดกึ่งกลาง แล้วหมุนมุมทั้ง 4 ของ bbox ตาม → ครอบด้วยกล่องแกนตั้ง (กล่องจะโตขึ้นเล็กน้อย — ระบุในรายงาน)
# ทำไม   : spec ข้อ 7 Paper 2 — สายพานจริงมีแสงเปลี่ยน/สั่น/มุมกล้องต่าง; ต้องรู้ว่าเงื่อนไขไหนทำให้ mAP ตกและ Improved ทนกว่าไหม
ROB = DS / "robustness"
test_imgs = sorted(p for p in (DS / "test" / "images").iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png"))
if QUICK_TEST:
    test_imgs = test_imgs[:60]

def rotate_boxes(rows, W, H, deg):
    # หมุนกล่อง (normalized xywh) รอบจุดกึ่งกลางภาพ แล้วครอบด้วยกล่องแกนตั้ง + clip ให้อยู่ในภาพ
    M = cv2.getRotationMatrix2D((W / 2, H / 2), deg, 1.0)
    out = []
    for c, x, y, w, h in rows:
        pts = np.array([[(x - w / 2) * W, (y - h / 2) * H], [(x + w / 2) * W, (y - h / 2) * H],
                        [(x + w / 2) * W, (y + h / 2) * H], [(x - w / 2) * W, (y + h / 2) * H]])
        pts = (M[:, :2] @ pts.T).T + M[:, 2]
        x1, y1 = np.clip(pts[:, 0].min(), 0, W), np.clip(pts[:, 1].min(), 0, H)
        x2, y2 = np.clip(pts[:, 0].max(), 0, W), np.clip(pts[:, 1].max(), 0, H)
        if x2 - x1 >= 2 and y2 - y1 >= 2:
            out.append((c, (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H))
    return out

def perturb(img, variant, cfg):
    if variant == "low_light" or variant == "over_exposure":
        return cv2.convertScaleAbs(img, alpha=cfg["alpha"], beta=cfg["beta"])
    if variant == "motion_blur":
        k = np.zeros((cfg["ksize"], cfg["ksize"]), dtype=np.float32)
        k[cfg["ksize"] // 2, :] = 1.0 / cfg["ksize"]
        return cv2.filter2D(img, -1, k)
    if variant == "tilt":
        H, W = img.shape[:2]
        M = cv2.getRotationMatrix2D((W / 2, H / 2), cfg["degrees"], 1.0)
        return cv2.warpAffine(img, M, (W, H), borderMode=cv2.BORDER_REPLICATE)
    if variant == "noise":
        noise = np.random.default_rng(SEED).normal(0, cfg["sigma"], img.shape)
        return np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
    raise ValueError(variant)

ROB_YAML = {}
for variant, cfg in NB06["robust"].items():
    vdir = ROB / variant
    marker = vdir / ".done"
    if not marker.exists() or QUICK_TEST:
        shutil.rmtree(vdir, ignore_errors=True)
        (vdir / "images").mkdir(parents=True)
        (vdir / "labels").mkdir(parents=True)
        for p in test_imgs:
            img = cv2.imread(str(p))
            out_img = perturb(img, variant, cfg)
            cv2.imwrite(str(vdir / "images" / p.name), out_img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            lbl = DS / "test" / "labels" / (p.stem + ".txt")
            rows = [tuple(float(v) if i else int(float(v)) for i, v in enumerate(ln.split()[:5])) for ln in lbl.read_text().splitlines() if ln.strip()]
            if variant == "tilt":
                H, W = img.shape[:2]
                rows = rotate_boxes(rows, W, H, cfg["degrees"])
            (vdir / "labels" / (p.stem + ".txt")).write_text("".join(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n" for c, x, y, w, h in rows), encoding="utf-8")
        marker.touch()
    y = {"path": str(vdir), "train": "images", "val": "images", "nc": NC, "names": {k: v for k, v in CLASSES.items()}, "note": f"robustness variant {variant}: {cfg}"}
    ROB_YAML[variant] = vdir / "data.yaml"
    with open(ROB_YAML[variant], "w", encoding="utf-8") as f:
        yaml.safe_dump(y, f, allow_unicode=True, sort_keys=False)
print(f"[OK] robustness sets: {list(NB06['robust'])} × {len(test_imgs)} ภาพ → {ROB.relative_to(PROJECT_ROOT)}")

rob_rows = []
for name in MODELS:
    clean = TEST[name]["summary"] if not QUICK_TEST else summarize(val(name, DATA_TEST, "test", split="test"))
    rob_rows.append({"model": name, "variant": "clean", "mAP50": clean["mAP50"], "mAP50-95": clean["mAP50-95"]})
    for variant, y in ROB_YAML.items():
        s = summarize(val(name, str(y), f"rob_{variant}"))
        rob_rows.append({"model": name, "variant": variant, "mAP50": s["mAP50"], "mAP50-95": s["mAP50-95"]})
rob = pd.DataFrame(rob_rows)
for name in MODELS:
    base = rob[(rob.model == name) & (rob.variant == "clean")].iloc[0]
    rob.loc[rob.model == name, "drop_mAP50"] = rob.loc[rob.model == name, "mAP50"] - base["mAP50"]
    rob.loc[rob.model == name, "drop_mAP50-95"] = rob.loc[rob.model == name, "mAP50-95"] - base["mAP50-95"]
rob_tab = rob.set_index(["model", "variant"])
print("\n— Robustness: mAP และ drop เทียบ clean (ค่าลบ = แย่ลง) —")
print(rob_tab.round(4).to_string())
rob_tab.to_csv(DIRS["reports"] / "06_robustness.csv", encoding="utf-8-sig")
RUN["robustness"] = rob.round(4).to_dict("records")

# --- กราฟ: heatmap ของ drop mAP50 (model × variant) + ตัวอย่างภาพ 1 ใบทุก variant ---
variants = ["clean", *NB06["robust"]]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [1.1, 2]})
mat = np.array([[rob_tab.loc[(n, v), "drop_mAP50"] for v in variants] for n in MODELS])
im = axes[0].imshow(mat, cmap="Reds_r", vmin=min(mat.min(), -0.01), vmax=max(0.0, mat.max()))
axes[0].set_xticks(range(len(variants)), variants, rotation=25, ha="right")
axes[0].set_yticks(range(len(MODELS)), list(MODELS))
axes[0].grid(False)
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        axes[0].text(j, i, f"{mat[i, j]:+.3f}", ha="center", va="center", fontsize=8.5, color="white" if mat[i, j] < mat.min() * 0.6 else viz.TEXT["primary"])
axes[0].set_title("mAP50 drop vs clean (test)", loc="left", fontsize=10.5)
fig.colorbar(im, ax=axes[0], fraction=0.05, pad=0.03)
sample = test_imgs[len(test_imgs) // 3]
tiles = [cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)] + [cv2.cvtColor(cv2.imread(str(ROB / v / "images" / sample.name)), cv2.COLOR_BGR2RGB) for v in NB06["robust"]]
h = min(t.shape[0] for t in tiles)
strip = np.concatenate([cv2.resize(t, (int(t.shape[1] * h / t.shape[0]), h)) for t in tiles], axis=1)
axes[1].imshow(strip)
axes[1].axis("off")
axes[1].set_title("example: " + " | ".join(variants), loc="left", fontsize=9.5)
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "06_robustness.png")
plt.show()
plt.close(fig)
print(f"[OK] {out.relative_to(PROJECT_ROOT)} | reports/06_robustness.csv")


In [ ]:
# === CELL 8 === Speed benchmark (Paper 3): FPS + ms/frame บน GPU (FP32 / FP16 quantize=16, NMS vs NMS-free) และ CPU + model size + trade-off
# ทำอะไร : predict ทีละภาพ (batch 1 = สถานการณ์ real-time) warm-up แล้ววัด n ภาพ → เฉลี่ย pre/infer/post จาก r.speed และ FPS จาก wall time
# ทำไม   : spec ข้อ 7 Paper 3 ต้องรายงาน FPS, latency, model size, mAP trade-off; half= ถูกแทนด้วย quantize=16 ใน ultralytics 8.4 (nb01)
bench_imgs = [str(p) for p in test_imgs[:NB06["speed"]["n_images"]]]
def bench(name, device, nms_flag, quant=None, n=None):
    imgs = bench_imgs[:n] if n else bench_imgs
    kw = dict(imgsz=NB06["imgsz"], device=device, nms=nms_flag, verbose=False, conf=DEPLOY[FINAL]["conf"])
    if quant:
        kw["quantize"] = quant
    m = MODELS[name]
    for p in imgs[:NB06["speed"]["warmup"]]:
        m.predict(p, **kw)
    sp, t0 = [], time.perf_counter()
    for p in imgs:
        sp.append(m.predict(p, **kw)[0].speed)
    wall = time.perf_counter() - t0
    pre, inf, post = (float(np.mean([s[k] for s in sp])) for k in ("preprocess", "inference", "postprocess"))
    return {"pre_ms": round(pre, 2), "infer_ms": round(inf, 2), "post_ms": round(post, 2), "latency_ms": round(pre + inf + post, 2),
            "fps_wall": round(len(imgs) / wall, 1), "n": len(imgs)}

speed_rows = []
configs = [("GPU FP32 NMS", 0, None, None), ("GPU FP32 NMS-free", 0, False, None), ("GPU FP16 NMS-free", 0, False, 16)] if DEVICE != "cpu" else []
for name in MODELS:
    for label, dev, nms_flag, quant in configs:
        try:
            speed_rows.append({"model": name, "config": label, **bench(name, dev, nms_flag, quant)})
        except Exception as e:
            print(f"[WARN] {name} {label} ล้มเหลว: {str(e)[:120]}")
    speed_rows.append({"model": name, "config": "CPU FP32 NMS-free", **bench(name, "cpu", False, None, n=NB06["speed"]["cpu_images"])})
size_mb = {name: round(p.stat().st_size / 1e6, 2) for name, p in EXPS.items()}
params_m = {name: round(sum(x.numel() for x in MODELS[name].model.parameters()) / 1e6, 2) for name in MODELS}
speed_tab = pd.DataFrame(speed_rows)
speed_tab["model_size_MB"] = speed_tab["model"].map(size_mb)
speed_tab["params_M"] = speed_tab["model"].map(params_m)
speed_tab["test_mAP50"] = speed_tab["model"].map(lambda n: round(TEST[n]["summary"]["mAP50"], 4))
speed_tab["test_mAP50-95"] = speed_tab["model"].map(lambda n: round(TEST[n]["summary"]["mAP50-95"], 4))
speed_tab = speed_tab.set_index(["model", "config"])
print("— Speed benchmark (batch 1, imgsz %d, conf %.2f) —" % (NB06["imgsz"], DEPLOY[FINAL]["conf"]))
print(speed_tab.to_string())
speed_tab.to_csv(DIRS["reports"] / "06_speed_benchmark.csv", encoding="utf-8-sig")
RUN["speed"] = speed_tab.reset_index().to_dict("records")
RUN["model_size_mb"], RUN["params_m"] = size_mb, params_m

fig, ax = plt.subplots(figsize=(9, 3.8))
lab = [f"{m}\n{c}" for m, c in speed_tab.index]
y = np.arange(len(lab))
ax.barh(y, speed_tab["pre_ms"], color="#b9b8b1", label="preprocess")
ax.barh(y, speed_tab["infer_ms"], left=speed_tab["pre_ms"], color=viz.SOURCE_COLORS["mendeley"], label="inference")
ax.barh(y, speed_tab["post_ms"], left=speed_tab["pre_ms"] + speed_tab["infer_ms"], color=viz.SOURCE_COLORS["roboflow"], label="postprocess")
for i, (lat, fps) in enumerate(zip(speed_tab["latency_ms"], speed_tab["fps_wall"])):
    ax.text(lat, i, f"  {lat:.1f} ms · {fps:.0f} FPS", va="center", fontsize=8, color=viz.TEXT["secondary"])
ax.set_yticks(y, lab, fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("ms per frame (batch 1)")
ax.margins(x=0.25)
ax.set_title(f"Latency breakdown — yolo26s ({params_m[FINAL]}M params, {size_mb[FINAL]} MB)", loc="left", fontsize=11)
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "06_speed_benchmark.png")
plt.show()
plt.close(fig)
print(f"[OK] {out.relative_to(PROJECT_ROOT)} | reports/06_speed_benchmark.csv")


In [ ]:
# === CELL 9 === Business KPI จำลอง (Paper 3): ผล/นาที, อัตราคัดแยกผิด, ประมาณการลด food loss — จากตัวเลขจริง + สมมติฐานใน NB06["kpi"]
# ทำอะไร : ใช้ FPS (GPU NMS-free ถ้ามี ไม่งั้น CPU) และ confusion matrix ของโมเดลสุดท้ายบน test คำนวณ KPI; ทุกสมมติฐานพิมพ์กำกับชัดเจน
# ทำไม   : spec ข้อ 3/7 ต้องการ business value — ตัวเลขนี้เป็น "การจำลองภายใต้สมมติฐาน" ไม่ใช่ผลวัดจริงจากโรงงาน (ต้องเขียนแบบนี้ในรายงาน)
K = NB06["kpi"]
sp = speed_tab.reset_index()
pref = sp[(sp.model == FINAL) & (sp.config == "GPU FP16 NMS-free")]
if pref.empty:
    pref = sp[(sp.model == FINAL) & (sp.config == "GPU FP32 NMS-free")]
if pref.empty:
    pref = sp[(sp.model == FINAL) & (sp.config == "CPU FP32 NMS-free")]
fps = float(pref["fps_wall"].iloc[0])
cfg_used = pref["config"].iloc[0]
capacity_bpm = fps * 60 * K["bananas_per_frame"]

M = CM[FINAL]["raw"]
matched = M[:NC, :NC].sum()
wrong_grade = matched - np.trace(M[:NC, :NC])
grade_error = wrong_grade / max(matched, 1)                 # จัดคลาสผิดในกล่องที่ตรวจเจอ
miss_rate = M[NC, :NC].sum() / max(M[:, :NC].sum(), 1)     # กล้วยจริงที่ไม่ถูกตรวจเจอ
false_alarm = M[:NC, NC].sum() / max(M[:NC, :].sum(), 1)   # กล่องที่ทายว่ามีกล้วยแต่ไม่มี
j_over = NAMES.index("overripe")
over_true_all = M[:, j_over].sum()                                          # overripe จริงทั้งหมด (รวมที่พลาดตรวจ)
missed_reject = (over_true_all - M[j_over, j_over]) / max(over_true_all, 1)   # overripe จริงที่ไม่ถูกจัดเป็น overripe (จัดผิดคลาส หรือพลาดตรวจ) = หลุดไปสายการผลิต
false_reject = M[j_over, :NC].sum() - M[j_over, j_over]                     # ไม่ใช่ overripe แต่ถูกจัดเป็น overripe (ทิ้งของดี)
false_reject_rate = false_reject / max(M[:NC, :NC].sum() - M[:NC, j_over].sum(), 1)   # หารด้วยกล้วยที่ตรวจเจอและไม่ใช่ overripe

monthly = K["line_speed_bpm"] * 60 * K["shift_hours"] * K["days_per_month"]
human_err = K["human_error_rate"]
model_err = float(grade_error)                # นับเฉพาะการจัดคลาสผิดของกล้วยที่ตรวจเจอ (กล้วยที่พลาดตรวจถือว่าไปสู่การตรวจซ้ำโดยคน)
misgraded_human, misgraded_model = monthly * human_err, monthly * model_err
loss_human = misgraded_human * K["loss_fraction_of_misgraded"]
loss_model = misgraded_model * K["loss_fraction_of_misgraded"]
saving_bananas = max(loss_human - loss_model, 0)
saving_thb = saving_bananas * K["cost_per_banana_thb"]

kpi = {
    "throughput": {"fps_measured": fps, "config": cfg_used, "capacity_bananas_per_min": round(capacity_bpm), "line_speed_bpm": K["line_speed_bpm"],
                   "keeps_up": bool(capacity_bpm >= K["line_speed_bpm"]), "utilization_pct": round(K["line_speed_bpm"] / max(capacity_bpm, 1e-9) * 100, 1)},
    "error_rates_test": {"grade_error_rate": round(float(grade_error), 4), "miss_rate": round(float(miss_rate), 4), "false_alarm_rate": round(float(false_alarm), 4),
                         "overripe_missed_reject_rate": round(float(missed_reject), 4), "false_reject_rate": round(float(false_reject_rate), 4)},
    "food_loss_simulation": {"monthly_bananas": int(monthly), "misgraded_human": int(misgraded_human), "misgraded_model": int(misgraded_model),
                             "food_loss_human": int(loss_human), "food_loss_model": int(loss_model),
                             "saving_bananas_per_month": int(saving_bananas), "saving_thb_per_month": round(saving_thb),
                             "assumptions": K},
}
print(f"=== Business KPI (โมเดลสุดท้าย: {FINAL}) — ตัวเลขจำลองภายใต้สมมติฐานใน NB06['kpi'] ===")
print(f"  Throughput : {fps:.0f} FPS ({cfg_used}) × {K['bananas_per_frame']} ผล/เฟรม → รองรับ {capacity_bpm:,.0f} ผล/นาที | สายพาน {K['line_speed_bpm']} ผล/นาที → "
      f"{'รองรับได้' if kpi['throughput']['keeps_up'] else 'ไม่ทัน'} (ใช้กำลัง {kpi['throughput']['utilization_pct']}%)")
print(f"  Error (test): จัดคลาสผิด {grade_error * 100:.2f}% ของกล้วยที่ตรวจเจอ | พลาดตรวจ {miss_rate * 100:.2f}% | false alarm {false_alarm * 100:.2f}%")
print(f"  QC          : overripe หลุดผ่าน (missed reject) {missed_reject * 100:.2f}% | ของดีถูกคัดทิ้ง (false reject) {false_reject_rate * 100:.2f}%")
print(f"  Food loss   : เดือนละ {monthly:,.0f} ผล | คัดผิดโดยคน {human_err * 100:.0f}% (สมมติ) = {misgraded_human:,.0f} ผล vs โมเดล {model_err * 100:.2f}% = {misgraded_model:,.0f} ผล")
print(f"                → food loss ลดลง ≈ {saving_bananas:,.0f} ผล/เดือน ≈ {saving_thb:,.0f} บาท/เดือน (ที่ {K['cost_per_banana_thb']} บาท/ผล, "
      f"{K['loss_fraction_of_misgraded'] * 100:.0f}% ของที่คัดผิดกลายเป็น loss)")
RUN["kpi"] = kpi


In [ ]:
# === CELL 10 === บันทึกทุกอย่าง: 06_evaluation.json, comparison table (csv + md), final_model ลง project.yaml + สรุป
# ทำอะไร : รวมตัวเลขทั้งหมดของ notebook นี้ลงไฟล์ และชี้โมเดลสุดท้าย + conf/iou/nms สำหรับ nb07
# ทำไม   : เล่มรายงานและ nb07 ต้องอ่านจากไฟล์เดียวกัน ไม่คัดลอกตัวเลขด้วยมือ
comp_full = comp.copy()
comp_full["test_mAP50_occlusion"] = [occ_tab.loc[(n, "occlusion"), "mAP50"] for n in comp_full.index]
comp_full["test_mAP50_NMSfree"] = [nms_tab.loc[(n, "full_test", "NMS-free (nms=False)"), "mAP50"] for n in comp_full.index]
comp_full["robust_min_drop_mAP50"] = [rob_tab.loc[n]["drop_mAP50"].min() for n in comp_full.index]
comp_full["latency_ms"] = [speed_tab.loc[n].iloc[-1]["latency_ms"] if DEVICE == "cpu" else speed_tab.loc[(n, "GPU FP32 NMS-free"), "latency_ms"] for n in comp_full.index]
comp_full["model_size_MB"] = [size_mb[n] for n in comp_full.index]
comp_full["best_epoch"] = [CONFIG["experiments"][n]["best_epoch"] for n in comp_full.index]
comp_full["final"] = [n == FINAL for n in comp_full.index]
comp_full.round(4).to_csv(DIRS["reports"] / "06_comparison_table.csv", encoding="utf-8-sig")
md = ["# Baseline vs Improved — test set", f"generated by 06_evaluate.ipynb on {datetime.now():%Y-%m-%d %H:%M} · final model = **{FINAL}** ({RUN['final_model']['reason']})", "",
      "| metric | " + " | ".join(comp_full.index) + " |", "|---|" + "---|" * len(comp_full.index)]
for col in comp_full.columns:
    md.append(f"| {col} | " + " | ".join(f"{v:.4f}" if isinstance(v, float) else str(v) for v in comp_full[col]) + " |")
md += ["", "## Confusion-matrix analysis", ""] + [f"- {l}" for n in CM for l in CM[n]["analysis"]] + ["", "## Learning-curve diagnosis", ""] + \
      [f"- {n}: {RUN['learning_curves'][n]['verdict']}" for n in RUN["learning_curves"]] + ["", "## Deploy threshold", "",
      f"- {FINAL}: conf {DEPLOY[FINAL]['conf']} (best mean F1 {DEPLOY[FINAL]['f1']:.4f}), NMS-free (nms=False), imgsz {NB06['imgsz']}", "",
      "## Business KPI (simulation — assumptions in reports/06_evaluation.json → kpi.food_loss_simulation.assumptions)", "",
      f"- throughput {kpi['throughput']['capacity_bananas_per_min']:,} bananas/min capacity vs line {K['line_speed_bpm']} → keeps up: {kpi['throughput']['keeps_up']}",
      f"- grade error {kpi['error_rates_test']['grade_error_rate'] * 100:.2f}% · overripe missed-reject {kpi['error_rates_test']['overripe_missed_reject_rate'] * 100:.2f}% · false reject {kpi['error_rates_test']['false_reject_rate'] * 100:.2f}%",
      f"- estimated food-loss reduction ≈ {kpi['food_loss_simulation']['saving_bananas_per_month']:,} bananas/month ≈ {kpi['food_loss_simulation']['saving_thb_per_month']:,} THB/month"]
(DIRS["reports"] / "06_comparison_table.md").write_text("\n".join(md) + "\n", encoding="utf-8")

CONFIG["final_model"] = {"name": FINAL, "best": CONFIG["experiments"][FINAL]["best"], "conf": DEPLOY[FINAL]["conf"], "iou": NB06["iou_default"],
                         "nms": False, "imgsz": NB06["imgsz"], "test_mAP50": round(float(comp.loc[FINAL, "mAP50"]), 4),
                         "test_mAP50_95": round(float(comp.loc[FINAL, "mAP50-95"]), 4), "reason": RUN["final_model"]["reason"], "chosen_by": "06_evaluate.ipynb"}
with open(CFG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)
RUN["finished_at"] = datetime.now().isoformat(timespec="seconds")
with open(DIRS["reports"] / "06_evaluation.json", "w", encoding="utf-8") as f:
    json.dump(RUN, f, ensure_ascii=False, indent=2, default=str)

print("— comparison table —")
print(comp_full.round(4).T.to_string())
print(f"\n[OK] reports/06_evaluation.json | 06_comparison_table.{{csv,md}} | figures/06_*.png ({len(list(DIRS['figures'].glob('06_*.png')))} ไฟล์)")
print(f"[OK] project.yaml → final_model = {FINAL} (conf {DEPLOY[FINAL]['conf']}, nms=False, imgsz {NB06['imgsz']})")
print("\n=== SUMMARY · Notebook 06 ===")
print(f"  final model: {FINAL} — test mAP50 {comp.loc[FINAL, 'mAP50']:.4f} / mAP50-95 {comp.loc[FINAL, 'mAP50-95']:.4f} | {RUN['final_model']['reason']}")
print(f"  ถัดไป: 07_video_inference.ipynb (ByteTrack + QC decision logic)")


## ผลลัพธ์ที่ควรเห็น
- CELL 2: `[TEST] baseline ...` / `[TEST] improved ...` + `[FINAL MODEL] <name> — เหตุผล: ...`
- CELL 3: heatmap 2 รูป + ข้อความวิเคราะห์ (คู่ที่สับสนมากสุด — คาด semi_ripe ↔ ripe)
- CELL 4: กราฟ 7 ช่อง + บรรทัดวินิจฉัย `→ UNDER-TRAINED / CONVERGED / OVERFIT`
- CELL 5: `[DEPLOY] conf threshold = x.xx`
- CELL 6–8: ตาราง occlusion / NMS vs NMS-free / robustness drop / speed
- CELL 9: KPI พร้อมคำว่า "สมมติ" ทุกจุดที่เป็นสมมติฐาน
- ไฟล์: `reports/06_*.{json,csv,md}`, `figures/06_*.png` (8 รูป), `datasets/banana/robustness/`, `project.yaml → final_model`

## ปัญหาที่อาจเจอ + วิธีแก้
| อาการ | สาเหตุ | วิธีแก้ |
|---|---|---|
| CELL 8 `GPU FP16 ... ล้มเหลว` | `quantize=16` ใช้ไม่ได้กับ build นี้ | ปกติ — ตารางจะข้ามแถวนั้น (nb01 บันทึกว่า half= ถูกแทนด้วย quantize=) |
| CELL 7 ช้า | val 5 variant × 2 โมเดล × 405 ภาพ | บน GPU ~3–5 นาที; ถ้าจำเป็นลด variant ใน `NB06["robust"]` |
| confusion matrix เป็น 0 ทั้งหมด | val ไม่ได้เปิด `plots=True` | CELL 2 เปิดไว้แล้ว — อย่าแก้ |
| ตัวเลข CELL 6/7 ต่างจาก CELL 2 เล็กน้อย | subset/ภาพต่างกัน | ปกติ |
